# GraphRAG en Producción: Comparativa y RAG Híbrido

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/graphrag/04-graphrag-en-produccion.ipynb)

Comparativa GraphRAG vs VectorRAG, implementación de RAG híbrido y checklist de producción.

## ¿Qué aprenderás?

- Medir latencia y coste de GraphRAG vs VectorRAG
- Implementar un sistema RAG híbrido inteligente con Claude
- Decidir automáticamente qué backend usar según el tipo de pregunta
- Aplicar el checklist de producción para despliegues reales

## Cuándo usar cada sistema

| Sistema | Fortalezas | Debilidades |
|---------|-----------|-------------|
| **VectorRAG** | Rápido, barato, semejanza semántica | No entiende relaciones, sin visión global |
| **GraphRAG Local** | Relaciones entre entidades, contexto rico | Más caro, más lento |
| **GraphRAG Global** | Síntesis de todo el corpus | Muy caro, lento |
| **Híbrido** | Lo mejor de cada uno | Complejidad operacional |

In [ ]:
# Instalar dependencias
# pip install anthropic numpy pandas matplotlib

import os
import time
import json
import random
from typing import Literal
from dataclasses import dataclass, field
from datetime import datetime

import anthropic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
MODO_SIMULADO = not ANTHROPIC_API_KEY

if not MODO_SIMULADO:
    cliente = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    print("Cliente Anthropic conectado (claude-haiku-4-5-20251001)")
else:
    cliente = None
    print("Modo simulado activado (sin ANTHROPIC_API_KEY)")

# Modelo de bajo coste para producción
MODELO = "claude-haiku-4-5-20251001"
PRECIO_INPUT_POR_M = 0.25   # USD por millón de tokens de entrada
PRECIO_OUTPUT_POR_M = 1.25  # USD por millón de tokens de salida

## 1. Función de benchmarking: latencia y coste por método

In [ ]:
@dataclass
class ResultadoBenchmark:
    """Resultado de una medición de rendimiento."""

    metodo: str
    pregunta: str
    respuesta: str
    latencia_ms: float
    tokens_entrada: int
    tokens_salida: int
    coste_usd: float
    calidad_estimada: float  # 0-1, estimada heurísticamente

    @property
    def coste_eur(self) -> float:
        return self.coste_usd * 0.93


def calcular_coste(tokens_entrada: int, tokens_salida: int) -> float:
    """Calcula el coste en USD dado el número de tokens."""
    return (
        tokens_entrada / 1_000_000 * PRECIO_INPUT_POR_M
        + tokens_salida / 1_000_000 * PRECIO_OUTPUT_POR_M
    )


# Simulación de base de conocimiento
BASE_CONOCIMIENTO = {
    "documentos": [
        "Anthropic desarrolla Claude. Dario Amodei es el CEO de Anthropic.",
        "OpenAI desarrolla ChatGPT. Sam Altman es el CEO de OpenAI.",
        "Google DeepMind desarrolla Gemini. Demis Hassabis lidera DeepMind.",
        "GraphRAG fue creado por Microsoft Research. Combina LLMs con grafos de conocimiento.",
        "El algoritmo de Leiden se usa en GraphRAG para detectar comunidades en grafos.",
    ],
    "grafo": {
        "entidades": ["Anthropic", "OpenAI", "Google DeepMind", "Microsoft", "Claude",
                      "ChatGPT", "Gemini", "GraphRAG", "Dario Amodei", "Sam Altman"],
        "relaciones": [
            ("Anthropic", "DESARROLLA", "Claude"),
            ("OpenAI", "DESARROLLA", "ChatGPT"),
            ("Google DeepMind", "DESARROLLA", "Gemini"),
            ("Microsoft", "CREÓ", "GraphRAG"),
            ("Dario Amodei", "LIDERA", "Anthropic"),
            ("Sam Altman", "LIDERA", "OpenAI"),
        ],
    },
}


def vectorrag_simulado(pregunta: str, n_chunks: int = 3) -> tuple[str, int, int]:
    """
    Simula VectorRAG: recupera chunks relevantes por similitud semántica simple.
    Retorna (contexto, tokens_entrada, tokens_salida).
    """
    # Simula recuperación semántica (en producción: embeddings + cosine similarity)
    chunks_relevantes = BASE_CONOCIMIENTO["documentos"][:n_chunks]
    contexto = "\n".join(chunks_relevantes)
    tokens_entrada = len(contexto.split()) * 1.3 + len(pregunta.split()) * 1.3 + 200
    tokens_salida = 80
    return contexto, int(tokens_entrada), tokens_salida


def graphrag_local_simulado(pregunta: str) -> tuple[str, int, int]:
    """
    Simula GraphRAG Local: explora subgrafo de entidades mencionadas.
    """
    entidades_relevantes = [
        e for e in BASE_CONOCIMIENTO["grafo"]["entidades"]
        if e.lower() in pregunta.lower()
    ]
    relaciones_relevantes = [
        f"{o} --[{t}]--> {d}" for o, t, d in BASE_CONOCIMIENTO["grafo"]["relaciones"]
        if o in entidades_relevantes or d in entidades_relevantes
    ]
    contexto = (
        f"Entidades: {', '.join(entidades_relevantes or ['sin entidades detectadas'])}\n"
        f"Relaciones: {'; '.join(relaciones_relevantes or ['sin relaciones'])}"
    )
    tokens_entrada = int((len(contexto.split()) + len(pregunta.split())) * 1.3 + 400)
    tokens_salida = 120
    return contexto, tokens_entrada, tokens_salida


def graphrag_global_simulado(pregunta: str) -> tuple[str, int, int]:
    """
    Simula GraphRAG Global: consulta resúmenes de todas las comunidades.
    """
    comunidades = [
        "Comunidad IA Asistentes: Anthropic (Claude), OpenAI (ChatGPT), Google (Gemini)",
        "Comunidad GraphRAG: Microsoft Research, Leiden, extracción de entidades",
        "Comunidad Liderazgo: Dario Amodei, Sam Altman, Demis Hassabis",
    ]
    contexto = "\n".join(comunidades)
    tokens_entrada = int((len(contexto.split()) + len(pregunta.split())) * 1.3 + 800)
    tokens_salida = 200
    return contexto, tokens_entrada, tokens_salida


def medir_benchmark(
    pregunta: str,
    metodo: Literal["vectorrag", "graphrag_local", "graphrag_global"],
    modo_simulado: bool = True,
) -> ResultadoBenchmark:
    """Ejecuta una consulta y mide latencia y coste."""

    # Obtener contexto según el método
    inicio = time.perf_counter()

    if metodo == "vectorrag":
        contexto, tok_in, tok_out = vectorrag_simulado(pregunta)
    elif metodo == "graphrag_local":
        contexto, tok_in, tok_out = graphrag_local_simulado(pregunta)
    else:
        contexto, tok_in, tok_out = graphrag_global_simulado(pregunta)

    if modo_simulado:
        # Simular latencia realista
        latencias = {"vectorrag": 180, "graphrag_local": 420, "graphrag_global": 1800}
        time.sleep(latencias[metodo] / 1000)  # Simular latencia
        respuesta = f"[Respuesta simulada - {metodo}] Basándome en el contexto: {contexto[:100]}..."
    else:
        prompt = f"Contexto:\n{contexto}\n\nPregunta: {pregunta}\n\nResponde de forma concisa."
        resp = cliente.messages.create(
            model=MODELO,
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
        )
        respuesta = resp.content[0].text
        tok_in = resp.usage.input_tokens
        tok_out = resp.usage.output_tokens

    latencia_ms = (time.perf_counter() - inicio) * 1000
    coste = calcular_coste(tok_in, tok_out)

    # Calidad heurística (en producción: evaluación con LLM)
    calidades = {"vectorrag": 0.65, "graphrag_local": 0.82, "graphrag_global": 0.90}
    calidad = calidades[metodo] + random.uniform(-0.05, 0.05)

    return ResultadoBenchmark(
        metodo=metodo,
        pregunta=pregunta,
        respuesta=respuesta,
        latencia_ms=round(latencia_ms, 1),
        tokens_entrada=tok_in,
        tokens_salida=tok_out,
        coste_usd=round(coste, 6),
        calidad_estimada=round(max(0, min(1, calidad)), 3),
    )


print("Funciones de benchmarking definidas.")
print(f"Modelo: {MODELO}")

## 2. HybridRAG: decide automáticamente qué backend usar

In [ ]:
class HybridRAG:
    """
    Sistema RAG híbrido que elige automáticamente el backend óptimo.
    
    Criterios de selección:
    - FACTUAL (simple): usa VectorRAG (más rápido y barato)
    - RELACIONAL (entidades y sus conexiones): usa GraphRAG Local
    - GLOBAL (síntesis, tendencias): usa GraphRAG Global
    """

    HERRAMIENTA_CLASIFICADOR = {
        "name": "clasificar_pregunta",
        "description": "Clasifica el tipo de pregunta RAG para seleccionar el backend óptimo.",
        "input_schema": {
            "type": "object",
            "properties": {
                "tipo": {
                    "type": "string",
                    "enum": ["factual", "relacional", "global"],
                    "description": (
                        "factual: pregunta sobre un hecho concreto; "
                        "relacional: pregunta sobre conexiones entre entidades; "
                        "global: pregunta sobre tendencias o visión de conjunto"
                    ),
                },
                "razon": {
                    "type": "string",
                    "description": "Breve justificación de la clasificación",
                },
                "confianza": {
                    "type": "number",
                    "minimum": 0,
                    "maximum": 1,
                },
            },
            "required": ["tipo", "razon", "confianza"],
        },
    }

    MAPEO_BACKEND = {
        "factual": "vectorrag",
        "relacional": "graphrag_local",
        "global": "graphrag_global",
    }

    def __init__(self, modo_simulado: bool = True):
        self.modo_simulado = modo_simulado
        self.historial: list[dict] = []

    def clasificar_pregunta(
        self, pregunta: str
    ) -> tuple[str, str, float]:
        """
        Clasifica la pregunta para elegir el backend.
        Retorna (tipo, razon, confianza).
        """
        if self.modo_simulado:
            return self._clasificar_heuristico(pregunta)

        resp = cliente.messages.create(
            model=MODELO,
            max_tokens=256,
            tools=[self.HERRAMIENTA_CLASIFICADOR],
            tool_choice={"type": "tool", "name": "clasificar_pregunta"},
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Clasifica esta pregunta para decidir el backend RAG óptimo:\n"
                        f"{pregunta}"
                    ),
                }
            ],
        )
        tool_result = next((b for b in resp.content if b.type == "tool_use"), None)
        if not tool_result:
            return "factual", "fallback", 0.5
        inp = tool_result.input
        return inp["tipo"], inp["razon"], inp["confianza"]

    def _clasificar_heuristico(self, pregunta: str) -> tuple[str, str, float]:
        """Clasificación simple por palabras clave (sin LLM)."""
        p = pregunta.lower()
        palabras_relacional = ["relación", "conexión", "trabaja", "fundó", "quién",
                                "entre", "con quién", "equipo", "colabora"]
        palabras_global = ["tendencia", "resumen", "panorama", "general", "todos",
                           "cuáles son", "qué temas", "visión", "comparativa"]

        if any(w in p for w in palabras_global):
            return "global", "Pregunta de síntesis o visión general", 0.80
        if any(w in p for w in palabras_relacional):
            return "relacional", "Pregunta sobre conexiones entre entidades", 0.85
        return "factual", "Pregunta sobre un hecho concreto", 0.75

    def consultar(self, pregunta: str) -> dict:
        """Clasifica la pregunta y ejecuta el backend apropiado."""
        tipo, razon, confianza = self.clasificar_pregunta(pregunta)
        backend = self.MAPEO_BACKEND[tipo]

        benchmark = medir_benchmark(pregunta, backend, modo_simulado=self.modo_simulado)

        registro = {
            "pregunta": pregunta,
            "tipo_detectado": tipo,
            "backend_usado": backend,
            "razon_clasificacion": razon,
            "confianza_clasificacion": confianza,
            "respuesta": benchmark.respuesta,
            "latencia_ms": benchmark.latencia_ms,
            "coste_usd": benchmark.coste_usd,
        }
        self.historial.append(registro)
        return registro


hybrid = HybridRAG(modo_simulado=MODO_SIMULADO)
print("HybridRAG listo.")
print(f"Estrategia de enrutamiento:")
for tipo, backend in HybridRAG.MAPEO_BACKEND.items():
    print(f"  {tipo} -> {backend}")

## 3. Demo: 3 tipos de preguntas

In [ ]:
PREGUNTAS_DEMO = [
    # Factual: dato concreto
    "¿Qué modelo de lenguaje desarrolla Anthropic?",
    # Relacional: conexiones entre entidades
    "¿Qué relación existe entre Dario Amodei y OpenAI?",
    # Global: síntesis del corpus
    "¿Cuáles son las tendencias generales en el sector de la IA generativa?",
]

resultados = []
print("=" * 70)
print("DEMO HybridRAG — Enrutamiento automático de preguntas")
print("=" * 70)

for i, pregunta in enumerate(PREGUNTAS_DEMO, 1):
    print(f"\n[{i}/3] Pregunta: {pregunta}")
    resultado = hybrid.consultar(pregunta)
    resultados.append(resultado)

    print(f"  Tipo detectado : {resultado['tipo_detectado']} "
          f"(confianza: {resultado['confianza_clasificacion']:.0%})")
    print(f"  Backend usado  : {resultado['backend_usado']}")
    print(f"  Razón          : {resultado['razon_clasificacion']}")
    print(f"  Latencia       : {resultado['latencia_ms']:.0f} ms")
    print(f"  Coste          : ${resultado['coste_usd']:.6f} USD")
    print(f"  Respuesta      : {resultado['respuesta'][:120]}...")

## 4. Benchmarking completo y visualización

In [ ]:
# Ejecutar benchmark completo: misma pregunta, los 3 métodos
PREGUNTA_BENCHMARK = "¿Qué empresas desarrollan asistentes de IA y quiénes las lideran?"

print(f"Benchmarking: '{PREGUNTA_BENCHMARK}'")
print()

benchmarks = []
for metodo in ["vectorrag", "graphrag_local", "graphrag_global"]:
    b = medir_benchmark(PREGUNTA_BENCHMARK, metodo, modo_simulado=MODO_SIMULADO)
    benchmarks.append(b)
    print(f"{metodo:20s}: {b.latencia_ms:6.0f} ms | ${b.coste_usd:.6f} | calidad: {b.calidad_estimada:.2f}")

# Tabla comparativa
df_bench = pd.DataFrame([
    {
        "Método": b.metodo,
        "Latencia (ms)": int(b.latencia_ms),
        "Tokens entrada": b.tokens_entrada,
        "Tokens salida": b.tokens_salida,
        "Coste USD": b.coste_usd,
        "Coste EUR": b.coste_eur,
        "Calidad (0-1)": b.calidad_estimada,
    }
    for b in benchmarks
])

print("\n=== Tabla comparativa ===")
print(df_bench.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
nombres = [b.metodo.replace("_", "\n") for b in benchmarks]
colores = ["#4A90D9", "#2ECC71", "#E67E22"]

# Latencia
axes[0].bar(nombres, [b.latencia_ms for b in benchmarks], color=colores, alpha=0.85)
axes[0].set_title("Latencia (ms)", fontweight="bold")
axes[0].set_ylabel("Milisegundos")

# Coste
axes[1].bar(nombres, [b.coste_usd * 1000 for b in benchmarks], color=colores, alpha=0.85)
axes[1].set_title("Coste (mUSD por consulta)", fontweight="bold")
axes[1].set_ylabel("mili-USD")

# Calidad estimada
axes[2].bar(nombres, [b.calidad_estimada for b in benchmarks], color=colores, alpha=0.85)
axes[2].set_title("Calidad estimada (0-1)", fontweight="bold")
axes[2].set_ylim(0, 1.1)

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle("Benchmarking: VectorRAG vs GraphRAG Local vs GraphRAG Global",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 5. Análisis coste-calidad del sistema híbrido

In [ ]:
# Simular distribución real de preguntas en producción
DISTRIBUCION_PREGUNTAS = {
    "factual": 0.60,     # 60% preguntas de hecho
    "relacional": 0.30,  # 30% sobre relaciones
    "global": 0.10,      # 10% síntesis global
}

# Costes medios por tipo de consulta (simulados)
COSTES_POR_TIPO = {
    "vectorrag": {"latencia_ms": 180, "coste_usd": 0.000040, "calidad": 0.65},
    "graphrag_local": {"latencia_ms": 420, "coste_usd": 0.000095, "calidad": 0.82},
    "graphrag_global": {"latencia_ms": 1800, "coste_usd": 0.000380, "calidad": 0.90},
}

BACKEND_POR_TIPO = {
    "factual": "vectorrag",
    "relacional": "graphrag_local",
    "global": "graphrag_global",
}

N_CONSULTAS_MES = 10_000

print("=" * 60)
print(f"ANÁLISIS ECONÓMICO — {N_CONSULTAS_MES:,} consultas/mes")
print("=" * 60)

coste_total_hibrido = 0.0
coste_total_graphrag_siempre = 0.0
coste_total_vector_siempre = 0.0
latencia_media_hibrido = 0.0
calidad_media_hibrido = 0.0

filas = []
for tipo, proporcion in DISTRIBUCION_PREGUNTAS.items():
    n = int(N_CONSULTAS_MES * proporcion)
    backend = BACKEND_POR_TIPO[tipo]
    metricas = COSTES_POR_TIPO[backend]

    coste_mes = n * metricas["coste_usd"]
    coste_total_hibrido += coste_mes
    coste_total_graphrag_siempre += n * COSTES_POR_TIPO["graphrag_local"]["coste_usd"]
    coste_total_vector_siempre += n * COSTES_POR_TIPO["vectorrag"]["coste_usd"]
    latencia_media_hibrido += proporcion * metricas["latencia_ms"]
    calidad_media_hibrido += proporcion * metricas["calidad"]

    filas.append({
        "Tipo pregunta": tipo,
        "% consultas": f"{proporcion:.0%}",
        "Consultas/mes": n,
        "Backend": backend,
        "Latencia (ms)": metricas["latencia_ms"],
        "Coste/mes USD": round(coste_mes, 2),
    })

df_economico = pd.DataFrame(filas)
print(df_economico.to_string(index=False))

print(f"\n{'=' * 60}")
print(f"Coste HÍBRIDO           : ${coste_total_hibrido:.2f}/mes")
print(f"Coste si TODO GraphRAG  : ${coste_total_graphrag_siempre:.2f}/mes")
print(f"Coste si TODO Vector    : ${coste_total_vector_siempre:.2f}/mes")
print(f"Ahorro vs todo GraphRAG : ${coste_total_graphrag_siempre - coste_total_hibrido:.2f}/mes "
      f"({(1 - coste_total_hibrido/coste_total_graphrag_siempre):.0%})")
print(f"Latencia media híbrido  : {latencia_media_hibrido:.0f} ms")
print(f"Calidad media híbrido   : {calidad_media_hibrido:.2f}/1.0")

## 6. Checklist de producción

### Infraestructura y almacenamiento

- [ ] **Base de datos de grafos**: Neo4j AuraDB (managed) o Neo4j self-hosted con backups automáticos
- [ ] **Vector store**: Qdrant, Weaviate o pgvector (según volumen y presupuesto)
- [ ] **Caché de consultas**: Redis para respuestas frecuentes (TTL: 1-24h según dominio)
- [ ] **Cola de indexación**: Celery + Redis o AWS SQS para procesar documentos de forma asíncrona

### Calidad y evaluación

- [ ] **Dataset de evaluación**: mínimo 50 preguntas con respuestas de referencia por tipo
- [ ] **Métricas**: RAGAS (faithfulness, answer_relevancy, context_recall)
- [ ] **A/B testing**: comparar respuestas híbridas vs un solo backend en prod
- [ ] **Fallback**: si GraphRAG tarda >2s, degradar a VectorRAG automáticamente

### Seguridad y privacidad

- [ ] **Sanitización de input**: prevenir injection via prompt
- [ ] **Control de acceso**: no exponer datos de un tenant a otro
- [ ] **PII detection**: detectar y anonimizar datos personales antes de indexar
- [ ] **Audit log**: registrar qué usuario consultó qué y cuándo

### Observabilidad

- [ ] **Métricas**: latencia p50/p95/p99, tasa de error, coste por consulta
- [ ] **Trazabilidad**: LangFuse, Langsmith o log estructurado por consulta
- [ ] **Alertas**: alerta si coste diario supera umbral (ej. $50/día)
- [ ] **Dashboard**: Grafana con paneles de latencia y coste por tipo de consulta

### Costes y optimización

- [ ] **Prompt caching**: usar Anthropic prompt caching para contextos repetidos (descuento 90%)
- [ ] **Batch API**: usar Anthropic Batch API para indexaciones nocturnas (descuento 50%)
- [ ] **Reindexación incremental**: solo procesar documentos nuevos/modificados
- [ ] **TTL en grafo**: eliminar nodos no accedidos en >90 días para controlar tamaño

## 7. Resumen del Bloque 30

### Lo que hemos construido

| Notebook | Contenido clave |
|----------|----------------|
| 01 - Grafos y Neo4j | Modelado de grafos, Cypher, PageRank |
| 02 - GraphRAG Microsoft | Pipeline completo, búsqueda global/local, estimación de costes |
| 03 - Extracción con LLM | `tool_use` de Claude, Pydantic, deduplicación con embeddings |
| 04 - Producción | Benchmarking, HybridRAG, análisis económico, checklist |

### Arquitectura de referencia para producción

```
Usuario
  │
  ▼
HybridRAG (clasificador)
  │
  ├── factual (60%) ──► VectorRAG ──► Qdrant
  ├── relacional (30%) ► GraphRAG Local ──► Neo4j
  └── global (10%) ───► GraphRAG Global ──► Resúmenes de comunidades
              │
              ▼
        Claude Haiku (generación final)
              │
              ▼
        Redis (caché TTL 1h)
```

### Recursos adicionales

- [GraphRAG Microsoft (GitHub)](https://github.com/microsoft/graphrag)
- [Paper original: "From Local to Global"](https://arxiv.org/abs/2404.16130)
- [Neo4j AuraDB (free tier)](https://neo4j.com/cloud/platform/aura-graph-database/)
- [Anthropic API Docs](https://docs.anthropic.com)
- [RAGAS — evaluación de RAG](https://github.com/explodinggradients/ragas)